# Zwaartekracht in de ORT

In het eerste notebook werd de SRT beschreven door alles te laten bewegen
door de ruimtetijd met de lichtsnelheid $c$. In dit notebook beschrijven
we het effect van zwaartekracht door de valsnelheid op te tellen bij de
ruimtesnelheid.

Dit notebook behandelt de éénmassa-oplossing. Uit deze ene uitbreiding
volgen alle basiseffecten van de zwaartekracht: tijddilatatie,
roodverschuiving, lichtafbuiging, baanprecissie, en de volledige
Schwarzschildmetriek.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / 'shared'))
from ort_core import *
from ort_plots import (c_local_profile, c_local_profile_interactive, spacetime_embedding_3d,
    light_speed_observers, spatial_stretching_comparison,
    orbital_precession_plot, light_deflection_diagram, photon_sphere_shadow, einstein_ring_plot, comparison_table)
import matplotlib.pyplot as plt
import math
import numpy as np
%matplotlib inline

---
# 1 Plaatsafhankelijke $c_{local}$

Nabij massa ontstaat een valsnelheid $v_{grav} = \sqrt{2GM/r}$, gericht naar
de massa. Door deze op te tellen bij $\vec{v}_{ruimte}$ wordt de snelheidsnorm:

$$v_{tijd}^2 + |\vec{v}_{ruimte} + \vec{v}_{grav}|^2 = c^2$$

De valsnelheid is de snelheid die een object vanuit oneindig in vrije val
bereikt op afstand $r$. Hoewel dit gelijk is aan de klassieke
ontsnappingssnelheid ($\frac{1}{2}mv^2 = GMm/r$), is het eigenlijk een
energieparameter: $\sqrt{2 E_{pot}/m_0}$. Toch werkt de vectoroptelling
exact — het komt overeen met het Gullstrand-Painlevé riviermodel, waarin
de ruimte zelf naar de massa toe stroomt met $v_{grav}$.

## 1.1 Uitwerking

Uitgeschreven met het inproduct:

$$v_{tijd}^2 + v_{ruimte}^2 + v_{grav}^2 + 2\,\vec{v}_{ruimte} \cdot \vec{v}_{grav} = c^2$$

### Twee lagen

**Isotroop (stilstaand, $v_{ruimte} = 0$):** het inproduct verdwijnt:

$$v_{tijd}^2 = c^2 - v_{grav}^2 = c_{local}^2$$

Zowel de tijddilatatie als de ruimtelijke uitrekking zijn isotroop —
onafhankelijk van richting.

**Anisotroop (bewegend, $v_{ruimte} \neq 0$):** de kruisterm
$2\,\vec{v}_{ruimte} \cdot \vec{v}_{grav}$ maakt het effect
richtingsafhankelijk. Tangentieel (loodrecht op $\vec{v}_{grav}$) is het
inproduct nul; radiaal (langs $\vec{v}_{grav}$) is het maximaal.

### De lokale ruimtetijdsnelheid

$$c_{local}(r) = c \cdot \sqrt{1 - \frac{2GM}{c^2 r}} = c \cdot \sqrt{1 - \frac{r_s}{r}} \qquad (1)$$

Voor een bewegend object hangt de beschikbare ruimtetijdsnelheid ook af van
richting — dat werken we uit in hoofdstuk 3.

In [ ]:
# c_local profiel nabij de eventhorizon
c_local_profile([SUN], lang='nl')
pass

---
## 1.2 De gradiënt van $c_{local}$

De afgeleide van formule (1) naar $r$:

$$\frac{dc_{local}}{dr} = \frac{GM}{c \cdot r^2 \cdot \sqrt{1 - r_s/r}} \qquad (2)$$

Vermenigvuldigd met $c$:

$$c \cdot \frac{dc_{local}}{dr} = \frac{GM}{r^2 \cdot \sqrt{1 - r_s/r}} = g \qquad (3)$$

Voor $r \gg r_s$ geeft dit Newtons $g = GM/r^2$. De gradiënt van $c_{local}$,
vermenigvuldigd met $c$, is de zwaartekrachtsversnelling $g$. Deze afleiding
geldt voor een stilstaand object ($v_{ruimte} = 0$) — het isotrope geval uit §1.1.

In [ ]:
# Verificatie: c · dc_local/dr = g
print("=== Gradiënt van c_local = zwaartekracht ===")
print()

# Aarde
print("--- Aarde (oppervlak) ---")
dc_dr_earth = EARTH.dc_local_dr(R_EARTH)
g_earth = EARTH.proper_acceleration(R_EARTH)
print(f"  dc_local/dr      = {dc_dr_earth:.6e} [1/s]")
print(f"  g                = {g_earth:.4f} m/s²")
print(f"  c · dc_local/dr  = {dc_dr_earth * C:.4f} m/s²")
print(f"  Verhouding       = {dc_dr_earth * C / g_earth:.15f}  (moet 1 zijn)")
print()

# Zon
print("--- Zon (oppervlak) ---")
dc_dr_sun = SUN.dc_local_dr(R_SUN)
g_sun = SUN.proper_acceleration(R_SUN)
print(f"  dc_local/dr      = {dc_dr_sun:.6e} [1/s]")
print(f"  g                = {g_sun:.4f} m/s²")
print(f"  c · dc_local/dr  = {dc_dr_sun * C:.4f} m/s²")
print(f"  Verhouding       = {dc_dr_sun * C / g_sun:.15f}")
print()

# Zwart gat (10 M☉) — sterk veld
BH = GravityModel(10 * M_SUN)
print("--- Zwart gat 10 M☉ (r = 1.1 r_s) ---")
r_bh = 1.1 * BH.rs
dc_dr_bh = BH.dc_local_dr(r_bh)
g_bh = BH.proper_acceleration(r_bh)
print(f"  dc_local/dr      = {dc_dr_bh:.6e} [1/s]")
print(f"  g                = {g_bh:.6e} m/s²")
print(f"  c · dc_local/dr  = {dc_dr_bh * C:.6e} m/s²")
print(f"  Verhouding       = {dc_dr_bh * C / g_bh:.15f}")
print()
print("De relatie c · dc_local/dr = g geldt EXACT, van zwak tot sterk veld.")

### Afleiding via de ART

De ART leidt $g$ als volgt af:

| Stap | ART | ORT ($v_{ruimte} = 0$) | Betekenis |
|------|-----|-----|-----------|
| 1. Metriek | $f(r) = 1 - r_s/r$ | $(c_{local}/c)^2$ | Vorm van de ruimtetijd |
| 2. Viersnelheid | $u^t = 1/\sqrt{f}$ | $c/c_{local}$ | Hoe snel de tijd tikt |
| 3. Christoffelsymbool | $\Gamma^r_{tt} = f \cdot GM/r^2$ | $dc_{local}/dr$ | Hoe snel het veld verandert |
| 4. Geodeetvergelijking | $a^r_{coord} = -GM/r^2$ | idem: $-GM/r^2$ | Coördinaatversnelling |
| 5. Frameprojectie | $\times 1/\sqrt{f}$ | $\times c/c_{local}$ | Van coördinaten naar lokaal |

Dezelfde uitkomst. De ART via differentiaalgeometrie, de ORT via de gradiënt van $c_{local}$.

---
# 2 Tijdeffecten

Voor een stilstaand object volgt de tijddilatatie direct uit $c_{local}$.

## 2.1 Gravitationele tijddilatatie

Op afstand $r$ van een massa heeft een klok de lokale ruimtetijdsnelheid
$c_{local} < c$. Als de klok stilstaat in de ruimte ($v_{ruimte} = 0$) gaat
alle $c_{local}$ naar de tijdrichting:

$$\frac{\tau}{t_\infty} = \frac{c_{local}}{c} = \sqrt{1 - \frac{r_s}{r}} \qquad (4)$$

Dit is de Schwarzschild-tijddilatatie. Hoe dichter bij de massa, hoe trager
de tijd.

In [ ]:
# Tijddilatatie op het aardoppervlak, GPS-hoogte, ISS-hoogte
print("=== Gravitationele tijddilatatie (Aarde) ===")
print(f"Schwarzschildstraal Aarde: r_s = {EARTH.rs:.4e} m = {EARTH.rs*1000:.4f} mm")
print()

locations = [
    ("Aardoppervlak", R_EARTH),
    ("ISS (408 km)", R_ISS),
    ("GPS (20.200 km)", R_GPS),
]
for name, r in locations:
    td = EARTH.time_dilation_factor(r)
    diff_per_day = (1 - td) * 86400e6  # microseconden per dag
    print(f"{name:25s}: τ/t∞ = {td:.15f}  (verschil: {diff_per_day:.3f} µs/dag)")

In [ ]:
# Newton vs ORT: tijddilatatie op GPS-hoogte
# Newton kent GEEN tijddilatatie — tijd is absoluut!
r_gps = R_EARTH + 20_200_000  # GPS-hoogte ~20.200 km
td_ort = EARTH.time_dilation_factor(r_gps)
drift_per_day_us = (1 - td_ort) * 86400 * 1e6  # µs per dag

print("=== Newton vs ORT: gravitationele tijddilatatie ===")
print(f"Klok op GPS-hoogte ({20200} km):")
print(f"  Newton:  Δt = 0 µs/dag      (tijd is absoluut!)")
print(f"  ORT:     Δt = {drift_per_day_us:+.2f} µs/dag  (klok loopt SNELLER)")
print()
print(f"Zonder correctie: GPS zou na 1 dag {abs(drift_per_day_us) * C * 1e-6:.0f} m afwijken!")
print(f"Na 1 week: {abs(drift_per_day_us) * 7 * C * 1e-6:.0f} m — navigatie onbruikbaar.")

---
## 2.2 Gravitationele roodverschuiving

Licht uitgezonden waar $c_{local}$ lager is, heeft een lagere frequentie — waar de tijd trager tikt, is de frequentie lager. Een verre waarnemer meet dus een roodverschuiving:

$$\frac{f_{obs}}{f_{emit}} = \frac{c_{local}(r_{emit})}{c_{local}(r_{obs})} = \frac{\sqrt{1 - r_s/r_{emit}}}{\sqrt{1 - r_s/r_{obs}}} \qquad (5)$$

In [ ]:
# Pound-Rebka experiment (1959): 22.5 m hoogteverschil
h = 22.5  # meter
r_bottom = R_EARTH
r_top = R_EARTH + h

z = EARTH.gravitational_redshift(r_bottom, r_top)
delta_f_over_f = -z  # roodverschuiving = negatief

# Benadering: Δf/f ≈ g·h/c²
g = G * M_EARTH / R_EARTH**2
approx = g * h / C**2

print("=== Pound-Rebka experiment ===")
print(f"Hoogte: {h} m")
print(f"Roodverschuiving z     = {z:.6e}")
print(f"|Δf/f|                 = {abs(delta_f_over_f):.6e}")
print(f"Benadering g·h/c²      = {approx:.6e}")
print(f"Gemeten (1959)         = (2.57 ± 0.26) ·10⁻¹⁵")

---
## 2.3 GPS: SRT + zwaartekracht gecombineerd

Een GPS-satelliet combineert twee effecten: de snelheid ten opzichte van het
aardoppervlak en de positie hoger in het zwaartekrachtsveld beïnvloeden beide
de kloktijd.

$$\frac{\tau}{t} = \frac{\sqrt{c_{local}^2 - v^2}}{c} \qquad (6)$$

| Component | Oorzaak | Effect |
|-----------|---------|--------|
| SRT | Snelheid satelliet (3870 m/s) | −7 µs/dag (langzamer) |
| Zwaartekracht | Hogere $c_{local}$ op GPS-hoogte | +45 µs/dag (sneller) |
| **Netto** | **Combinatie** | **+38 µs/dag (sneller)** |

In [ ]:
# GPS correctie: combinatie SRT + zwaartekracht
v_gps = 3870  # m/s (GPS satelliet snelheid)

# Aardoppervlak (in rust)
td_surface = EARTH.combined_time_dilation(R_EARTH, 0)

# GPS satelliet (bewegend op hoogte)
td_gps = EARTH.combined_time_dilation(R_GPS, v_gps)

# Verschil in microseconden per dag
diff_us_per_day = (td_gps - td_surface) * 86400 * 1e6

# Afzonderlijke componenten
grav_only = (EARTH.time_dilation_factor(R_GPS) - EARTH.time_dilation_factor(R_EARTH)) * 86400 * 1e6
srt_only = (math.sqrt(1 - (v_gps/C)**2) - 1) * 86400 * 1e6

print("=== GPS Correctie ===")
print(f"Alleen zwaartekracht:   {grav_only:+.2f} µs/dag")
print(f"Alleen SRT (snelheid):  {srt_only:+.2f} µs/dag")
print(f"Gecombineerd (formule): {diff_us_per_day:+.2f} µs/dag")
print(f"\nVerwacht netto:         +38 µs/dag")

---
## 2.4 De eventhorizon ($c_{local} = 0$)

Bij $r = r_s$ wordt $v_{grav} = c$ en $c_{local} = 0$. De consequenties:

1. **Niets kan bewegen** — niet door de ruimte, niet door de tijd
2. **Klokken staan stil** — $v_{tijd} = 0$ voor een stilstaand object
3. **Licht kan niet ontsnappen** — uitgaand licht heeft $v = c - v_{grav} = 0$

Een vrijvallend object merkt niets bijzonders: $\vec{v}_{ruimte} = \vec{v}_{grav}$, dus $v_{tijd} = c$. Het steekt de horizon over in eindige eigen tijd — maar het licht dat het naar buiten stuurt staat stil.

In [ ]:
# Zwart gat van 10 zonsmassa's
BH_10 = GravityModel(10 * M_SUN)
print(f"=== Zwart gat van 10 M☉ ===")
print(f"r_s = {BH_10.rs:.3e} m = {BH_10.rs/1000:.2f} km")
print()

# c_local op verschillende afstanden van de horizon
factors = [10.0, 5.0, 2.0, 1.5, 1.1, 1.01, 1.001, 1.0]
print(f"{'r/r_s':>8s}  {'c_local/c':>12s}  {'c_local (m/s)':>15s}")
print("-" * 40)
for f in factors:
    r = f * BH_10.rs
    cl = BH_10.c_local(r)
    print(f"{f:8.3f}  {cl/C:12.6f}  {cl:15.0f}")

---
# 3 Ruimtelijke effecten

> **Kernprincipe**: de ontsnappingssnelheid $\vec{v}_{grav}$ is een vector — gericht
> naar de massa. Voor een stilstaand object is het effect isotroop (tijddilatatie),
> maar voor een bewegend object hangt het af van de hoek: het inproduct
> $\vec{v}_{ruimte} \cdot \vec{v}_{grav}$ bepaalt de ruimtelijke anisotropie.

## 3.1 Ruimtelijke uitrekking

### Richting-afhankelijke effecten

Het volledige snelheidsbudget is:

$$v_{tijd}^2 + v_{ruimte}^2 + v_{grav}^2 - 2\,\vec{v}_{ruimte} \cdot \vec{v}_{grav} = c^2$$

Het inproduct $\vec{v}_{ruimte} \cdot \vec{v}_{grav} = |v_{ruimte}| \cdot |v_{grav}| \cdot \cos\alpha$, met $\alpha$ de hoek tussen bewegingsrichting en $\vec{v}_{grav}$.

**Tangentieel** ($\alpha = 90°$, loodrecht op $\vec{v}_{grav}$): $\cos\alpha = 0$, inproduct verdwijnt.

$$v_{tang}^2 + v_{grav}^2 + v_{tijd}^2 = c^2 \qquad \Rightarrow \qquad v_{tang,max} = c_{local}$$

**Radiaal naar buiten** ($\alpha = 180°$): $(v_{ruimte} + v_{grav})^2 + v_{tijd}^2 = c^2$

$$v_{radiaal} = c - v_{grav} \quad \text{(voor een foton)}$$

**Radiaal naar binnen** ($\alpha = 0°$): $(v_{ruimte} - v_{grav})^2 + v_{tijd}^2 = c^2$

$$v_{radiaal} = c + v_{grav} \quad \text{(voor een foton)}$$

Dit is het **rivier-model**: de ruimte stroomt als het ware naar binnen met $v_{grav}$. Uitgaand licht zwemt tegen de stroom in, inkomend licht wordt meegevoerd.

### Consequenties voor de ruimtelijke uitrekking

De ruimtelijke uitrekking is **anisotroop** — alleen in de radiale richting:

$$dl = \frac{dr}{\sqrt{1 - r_s/r}} = dr \cdot \frac{c}{c_{local}} \qquad (30)$$

Tangentieel is er **geen** uitrekking: de omtrek op coördinaat $r$ is gewoon $2\pi r$.

Dit volgt direct uit de vectoraard van $\vec{v}_{grav}$:
- **Radiaal**: $\vec{v}_{ruimte}$ en $\vec{v}_{grav}$ zijn parallel → lineaire interactie → extra effect
- **Tangentieel**: $\vec{v}_{ruimte}$ en $\vec{v}_{grav}$ zijn loodrecht → kwadratische optelling → alleen $c_{local}$

### Beide diagonaalcomponenten

Samen met de tijdcomponent uit hoofdstuk 2:

| Component | Schwarzschild | ORT |
|-----------|--------------|-----|
| Tijd ($g_{tt}$) | $1 - r_s/r$ | $(c_{local}/c)^2$ |
| Ruimte ($g_{rr}$) | $(1 - r_s/r)^{-1}$ | $(c/c_{local})^2$ |
| Hoek ($g_{\phi\phi}$) | $r^2$ | $r^2$ |

De ORT reproduceert de volledige Schwarzschildmetriek. $g_{tt} \cdot g_{rr} = 1$ — de twee componenten zijn elkaars inverse. Eén principe ($\vec{v}_{grav}$ als vector) bepaalt alle drie.

### Wat zien de twee waarnemers?

- Een **lokale waarnemer** meet altijd $c$ in alle richtingen (equivalentieprincipe).
- Een **verre waarnemer** ziet licht vertragen nabij massa — maar **niet in alle richtingen gelijk**:

| Richting | Coördinaatlichtsnelheid | Verklaring |
|----------|------------------------|------------|
| Tangentieel | $c\sqrt{1 - r_s/r} = c_{local}$ | Alleen tijddilatatie |
| Radiaal (naar buiten) | $c(1 - \sqrt{r_s/r})$ | Tegen $\vec{v}_{grav}$ in |
| Radiaal (naar binnen) | $c(1 + \sqrt{r_s/r})$ | Mee met $\vec{v}_{grav}$ |

De anisotropie volgt direct uit de vectoraard van $\vec{v}_{grav}$: tangentieel staat $\vec{v}_{ruimte}$ loodrecht op $\vec{v}_{grav}$ (inproduct nul), radiaal zijn ze parallel (inproduct maximaal).

Dit komt overeen met het Gullstrand-Painlevé-model (het "rivier-model") uit de ART, waarin de ruimte naar de massa toe stroomt met snelheid $v_{grav}$.

In [ ]:
# Coördinaatlichtsnelheid: lokale vs verre waarnemer, ART vs ORT
light_speed_observers(lang='nl')
pass

In [ ]:
# Ruimtelijke uitrekking nabij de Zon en een 10 M☉ zwart gat
print("=== Ruimtelijke uitrekking ===")
print()
print("--- Zon (oppervlak) ---")
stretch_sun = SUN.spatial_stretching(R_SUN)
print(f"Uitrekkingsfactor: {stretch_sun:.10f}")
print(f"Extra lengte per km: {(stretch_sun - 1) * 1000:.6f} m")
print()
print("--- 10 M☉ zwart gat ---")
for f in [10.0, 3.0, 1.5, 1.1, 1.01]:
    r = f * BH_10.rs
    s = BH_10.spatial_stretching(r)
    print(f"  r = {f:.2f} r_s:  uitrekking = {s:.6f}  (1 km → {s:.6f} km)")

### Flamm-paraboloïde

De ruimtelijke uitrekking wordt traditioneel gevisualiseerd als het **Flamm-paraboloïde**: een 2D inbedding van het equatoriale vlak in 3D. De extra dimensie (z-as) is een wiskundig hulpmiddel — geen fysieke richting — die toont hoeveel de **radiale** afstand uitgerekt is.

Omdat de ORT dezelfde radiale uitrekking ($g_{rr} = (c/c_{local})^2$) en dezelfde tangentiële geometrie ($g_{\phi\phi} = r^2$, omtrek = $2\pi r$) geeft als de Schwarzschildmetriek, is het Flamm-paraboloïde ook de juiste visualisatie van de ORT.

In [ ]:
# 3D inbedding van de ruimtelijke geometrie (Flamm-paraboloïde) — ART-visualisatie
spacetime_embedding_3d(lang='nl')
pass

### Visualisatie: Newton vs ORT/ART

De grafieken hieronder vergelijken de radiale eigenafstand en de omtrek:

- **Links — eigenafstand (radiaal)**: ORT en ART zijn identiek; beide voorspellen meer radiale afstand dan Newton. Nabij $r_s$ rekt de ruimte radiaal uit — dit is het effect dat het Flamm-paraboloïde visualiseert.
- **Rechts — fysieke omtrek**: alle drie voorspellen $C = 2\pi r$ — er is geen tangentiële uitrekking. De omtrek wordt niet beïnvloed door de massa.

Bij $r \gg r_s$ komen alle lijnen samen — vlakke ruimte. Het verschil verschijnt alleen nabij massa, en alleen radiaal.

In [ ]:
# Vergelijking radiale eigenafstand en fysieke omtrek: Newton, ART, ORT
spatial_stretching_comparison(lang='nl')
pass

---
## 3.2 Lichtafbuiging

Licht dat langs een massa scheert wordt afgebogen. De afbuiging komt uit twee effecten:

1. **Tijddilatatie** ($c_{local} < c$): licht beweegt lokaal trager → buigt af als in een brekend medium
2. **Radiale uitrekking** (inproduct $\vec{v}_{ruimte} \cdot \vec{v}_{grav}$): de radiale component van het lichtpad ervaart extra vertraging

Beide bijdragen zijn even groot. De effectieve brekingsindex voor een radiaal lichtpad:

$$n_{eff} = \left(\frac{c}{c_{local}}\right)^2 = \frac{1}{1 - r_s/r} \qquad (31)$$

De totale afbuighoek:

$$\alpha = \frac{2r_s}{b} = \frac{4GM}{bc^2} \qquad (32)$$

Dit is het resultaat van de ART (Einstein 1915: 1.75").
Soldner (1801) en Einstein (1911) vonden de **helft**: alleen het tijdeffect, zonder de radiale uitrekking.

In [ ]:
# Lichtafbuiging diagram
fig = light_deflection_diagram(lang='nl')
plt.show()

In [ ]:
# Lichtafbuiging door de Zon (b = R_zon)
alpha_arcsec = SUN.light_deflection_arcsec(R_SUN)
alpha_half = SUN.half_light_deflection(R_SUN) * (180/math.pi) * 3600

print("=== Lichtafbuiging door de Zon ===")
print(f"Impactparameter b = R_zon = {R_SUN:.3e} m")
print(f"Soldner/Einstein 1911 (halve waarde): {alpha_half:.4f}\"")
print(f"ORT / ART (volledige):    {alpha_arcsec:.4f}\"")
print(f"Gemeten (Eddington 1919):             1.75 ± 0.06\"")

In [ ]:
# Newton vs ORT: lichtafbuiging door de Zon
# Newton/Soldner (1801) voorspelt de HELFT van het juiste antwoord!
alpha_newton = SUN.half_light_deflection(R_SUN) * (180 / math.pi) * 3600
alpha_ort = SUN.light_deflection_arcsec(R_SUN)

print("=== Newton vs ORT: lichtafbuiging door de Zon ===")
print(f"  Newton/Soldner (1801):  {alpha_newton:.4f}\"  (alleen tijdkromming)")
print(f"  ORT/Einstein (1915):    {alpha_ort:.4f}\"  (tijd + ruimtekromming)")
print(f"  Eddington (1919):       1.75 ± 0.06\"  (gemeten!)")
print()
print(f"Newton voorspelt exact de HELFT: {alpha_newton/alpha_ort:.3f}×")
print(f"De ontbrekende helft komt van ruimtekromming — iets dat Newton niet kent.")

---
## 3.3 Baanprecissie

In Newton levert de effectieve potentiaal een gesloten ellips — geen precessie.
In de ORT komt er een extra term bij door de radiale uitrekking:

$$V_{GR}(r) = -\frac{GM}{r} + \frac{L^2}{2r^2} - \frac{GML^2}{r^3 c^2} \qquad (34)$$

Net als bij lichtafbuiging komt de precessie uit twee bijdragen:
- **Tijddilatatie** ($g_{tt}$): variërende kloksnelheid
- **Radiale uitrekking** ($g_{rr}$): het inproduct $\vec{v}_{ruimte} \cdot \vec{v}_{grav}$ beïnvloedt de radiale component van de baan

Samen:

$$\Delta\varphi = \frac{3\pi r_s}{a(1-e^2)} \qquad \text{per omloop} \qquad (35)$$

### Mercurius — de klassieke test

Voor Mercurius: **$\Delta\varphi$ = 42.98"/eeuw** — exact de waargenomen waarde (Le Verrier 1859, Einstein 1915).

| Planeet   | Δφ/omloop (") | Omlopen/eeuw | Δφ/eeuw (") | Waargenomen (") |
|-----------|--------------|--------------|-------------|-----------------|
| Mercurius | 0.1035       | 415.2        | 42.98       | 43.0            |
| Venus     | 0.0053       | 162.5        | 0.86        | 8.6*            |
| Aarde     | 0.0038       | 100.0        | 0.38        | 3.8*            |

*\* Waargenomen waarden bevatten ook gravitationele invloeden van andere planeten.*

In [ ]:
# Baanprecissie van Mercurius
prec_per_orbit = SUN.orbital_precession_arcsec(A_MERCURY, E_MERCURY)
prec_per_century = SUN.orbital_precession_arcsec_century(A_MERCURY, E_MERCURY, T_MERCURY)

print("=== Baanprecissie van Mercurius ===")
print(f"Halve grote as a  = {A_MERCURY:.4e} m")
print(f"Excentriciteit e  = {E_MERCURY}")
print(f"Omlooptijd        = {T_MERCURY/86400:.3f} dagen")
print(f"\nΔφ per omloop      = {prec_per_orbit:.4f}\"")
print(f"Δφ per eeuw        = {prec_per_century:.2f}\"")
print(f"Waargenomen       = 43.0\"")

In [ ]:
# Newton vs ORT: baanprecissie van Mercurius
# Newton voorspelt gesloten ellipsen — 0 extra precissie!
prec_ort = SUN.orbital_precession_arcsec(A_MERCURY, E_MERCURY)
prec_per_century = prec_ort * (100 * 365.25 * 86400) / T_MERCURY

print("=== Newton vs ORT: baanprecissie van Mercurius ===")
print(f"  Newton:           0.00\"/eeuw   (ellips sluit exact!)")
print(f"  ORT/Einstein:    {prec_per_century:.2f}\"/eeuw")
print(f"  Waargenomen:     43.0 ± 0.1\"/eeuw  (Le Verrier, 1859)")
print()
print(f"Dit onverklaarde verschil was 56 jaar lang een raadsel.")
print(f"Einstein loste het op in 1915 — zijn eerste bevestiging van de ART.")

In [ ]:
# Precissie-plot voor Mercurius
fig = orbital_precession_plot(SUN, A_MERCURY, E_MERCURY, n_orbits=5, lang='nl')
plt.show()

---
# 4 Het complete beeld

> **Kernprincipe**: alles beweegt met $c$ door de ruimtetijd. Nabij massa gaat een
> deel naar de ontsnappingssnelheid $\vec{v}_{grav}$, gericht naar de massa. Voor een
> stilstaand object is het effect isotroop (tijddilatatie), voor een bewegend object
> hangt het af van de richting (ruimtelijke anisotropie via het inproduct).

Vijf effecten uit één principe:
1. **Klokken lopen langzamer** waar $c_{local}$ lager is → tijddilatatie (isotroop)
2. **De ruimte rekt radiaal uit** door het inproduct $\vec{v}_{ruimte} \cdot \vec{v}_{grav}$ → anisotrope uitrekking
3. **Licht buigt af** door tijddilatatie + radiale uitrekking (elk de helft)
4. **Licht verschuift naar lagere frequentie** → roodverschuiving
5. **Bij $v_{grav} = c$ stopt alles** → eventhorizon

---
## 4.1 De Schwarzschildverbinding

De Schwarzschildmetriek voor een bolvormige massa:

$$ds^2 = \left(1 - \frac{r_s}{r}\right) c^2 dt^2 - \left(1 - \frac{r_s}{r}\right)^{-1} dr^2 - r^2 d\Omega^2$$

De ORT reproduceert **alle drie** de componenten:

| | Schwarzschild | ORT | Herkomst |
|---|---|---|---|
| **Tijd**: $g_{tt}$ | $1 - r_s/r$ | $(c_{local}/c)^2$ | $v_{ruimte} = 0$: inproduct verdwijnt |
| **Ruimte**: $g_{rr}$ | $(1 - r_s/r)^{-1}$ | $(c/c_{local})^2$ | $\vec{v}_{ruimte} \parallel \vec{v}_{grav}$: lineaire interactie |
| **Hoek**: $g_{\phi\phi}$ | $r^2$ | $r^2$ | $\vec{v}_{ruimte} \perp \vec{v}_{grav}$: inproduct nul |

De overeenkomst is niet bij benadering — het is een **exacte gelijkheid**.
De vectoraard van $\vec{v}_{grav}$ bepaalt alle drie de metriekcomponenten.

In [ ]:
# Numerieke verificatie: g_tt en g_rr uit Schwarzschild vs uit c_local/c
print("=== Schwarzschildverificatie: g_tt en g_rr uit c_local ===")
print()
print(f"{'r/r_s':>8s}  {'g_tt (Schw)':>12s}  {'(c_l/c)²':>12s}  {'g_rr (Schw)':>12s}  {'(c/c_l)²':>12s}  {'match':>6s}")
print("-" * 70)
for f in [100.0, 10.0, 5.0, 3.0, 2.0, 1.5, 1.1, 1.01]:
    r = f * BH_10.rs
    # Schwarzschild metriekcomponenten
    g_tt_schw = 1 - BH_10.rs / r
    g_rr_schw = 1 / (1 - BH_10.rs / r)
    # ORT: uit c_local
    cl = BH_10.c_local(r)
    g_tt_ort = (cl / C) ** 2
    g_rr_ort = (C / cl) ** 2
    match = "✓" if abs(g_tt_schw - g_tt_ort) < 1e-12 and abs(g_rr_schw - g_rr_ort) < 1e-10 else "✗"
    print(f"{f:8.2f}  {g_tt_schw:12.8f}  {g_tt_ort:12.8f}  {g_rr_schw:12.8f}  {g_rr_ort:12.8f}  {match:>6s}")
print()
print("Eén principe (variërende c_local) → beide Schwarzschild-componenten: EXACT.")

---
## 4.2 Het patroon

| Regime | Snelheidsbudget | Wat volgt |
|--------|----------------|-----------|
| Geen massa | $v_{ruimte}^2 + v_{tijd}^2 = c^2$ | SRT: tijddilatatie, $E=mc^2$, Lorentz |
| Nabij massa, stilstaand | $v_{tijd} = c_{local}$ (isotroop) | Gravitationele tijddilatatie, roodverschuiving |
| Nabij massa, tangentieel | inproduct = 0 → $v_{tang,max} = c_{local}$ | Geen tangentiële uitrekking, omtrek = $2\pi r$ |
| Nabij massa, radiaal | inproduct ≠ 0 → $v_{rad} = c \pm v_{grav}$ | Radiale uitrekking, Shapiro-vertraging |
| Vrije val | $\vec{v}_{ruimte} = \vec{v}_{grav}$ → $v_{tijd} = c$ | Geen gravitationele tijddilatatie |
| Eventhorizon | $v_{grav} = c$ → $c_{local} = 0$ | Niets beweegt, Zijn = 0 |

Eén principe — "alles beweegt met $c$, nabij massa gaat $\vec{v}_{grav}$ daar vanaf" — beschrijft
zowel speciale relativiteit als alle zwaartekrachtseffecten:

$$v_{tijd}^2 + (\vec{v}_{ruimte} - \vec{v}_{grav})^2 = c^2$$

---
## 4.3 Scope: wat dit notebook beschrijft

**Beschreven (hoofdstuk 1–4):**
- Tijddilatatie, roodverschuiving, lichtafbuiging, baanprecissie
- Eventhorizon, ruimtelijke uitrekking, Schwarzschildmetriek
- Invariantie van Zijn in een zwaartekrachtsveld

**Vervolg (notebooks 03–05):**
- Reissner-Nordström, Shapiro-vertraging, geodetische precessie (NB 03)
- Fotonsfeer, zwaartekrachtsgolven, Kerr-metriek (NB 03)
- Kosmologie: ons heelal als binnenkant van een zwart gat (NB 04)
- Kwantummechanica en het Zijn-principe (NB 05)

---
## 4.4 Invariantie van Zijn in een zwaartekrachtsveld

In de SRT: $\text{Zijn} = m_0 \cdot c$. Nabij massa:

$$\text{Zijn} = m_0 \cdot c_{local} \qquad (38)$$

Consequenties:
- **Massa neemt NIET toe** in zwaartekracht — het is $c_{local}$ die daalt
- **Nabij andere zijnden daalt je Zijn** — anti-holistisch principe
- **Eventhorizon: Zijn = 0** — $c_{local} = 0 \Rightarrow m_0 \cdot 0 = 0$

Gravitationele bindingsenergie:

$$E_{binding} = m_0 c^2 \left(1 - \sqrt{1 - \frac{r_s}{r}}\right) \qquad (39)$$

In [ ]:
# Zijn nabij de Aarde en een zwart gat
m0 = 1.0  # kg referentiemassa
print("=== Invariantie van Zijn ===")
print(f"In vlakke ruimtetijd: Zijn = m₀ · c = {m0 * C:.6e} kg·m/s")
print()

print("--- Nabij de Aarde ---")
cl_earth = EARTH.c_local(R_EARTH)
zijn_earth = m0 * cl_earth
print(f"c_local(R_aarde)  = {cl_earth:.6f} m/s")
print(f"Zijn(R_aarde)     = {zijn_earth:.6e} kg·m/s")
print(f"Zijn/Zijn_∞       = {cl_earth/C:.15f}")
print()

print("--- Nabij 10 M☉ zwart gat ---")
for f in [10.0, 2.0, 1.1, 1.01]:
    r = f * BH_10.rs
    cl = BH_10.c_local(r)
    print(f"  r = {f:.2f} r_s:  Zijn = {m0 * cl:.6e} kg·m/s  ({cl/C:.6f} c)")

print()
# Bindingsenergie
E_bind = m0 * C**2 * (1 - EARTH.time_dilation_factor(R_EARTH))
print(f"Bindingsenergie 1 kg op aardoppervlak: {E_bind:.6e} J")

---

## Samenvatting

Alle basale zwaartekrachtseffecten volgen uit één uitbreiding:

$$v_{tijd}^2 + (\vec{v}_{ruimte} - \vec{v}_{grav})^2 = c^2$$

met $\vec{v}_{grav}$ de ontsnappingssnelheid, radiaal naar de massa gericht, en $c_{local} = c \cdot \sqrt{1 - r_s/r}$.

| # | Effect | Formule | Bevestiging |
|---|--------|---------|-------------|
| 1.1 | Ontsnappingssnelheid | $v_{grav} = c \cdot \sqrt{r_s/r}$ | Formule (29) |
| 1.2 | Gradiënt = zwaartekracht | $dc_{local}/dr = g/c$ | Formule (23c) |
| 2.1 | Tijddilatatie (isotroop) | $v_{tijd} = c_{local}$ voor stilstaand object | GPS, atoomklokken |
| 2.2 | Roodverschuiving | $f_{obs}/f_{emit} = c_{local,e}/c_{local,o}$ | Pound-Rebka |
| 2.3 | GPS-correctie | $v_{tijd} = \sqrt{c_{local}^2 - v^2}$ | Dagelijkse verificatie |
| 2.4 | Eventhorizon | $v_{grav} = c$ → $c_{local} = 0$ | EHT M87*, Sgr A* |
| 3.1 | Ruimtelijke uitrekking | Anisotroop: radiaal via inproduct $\vec{v}_{ruimte} \cdot \vec{v}_{grav}$ | Cassini ($\gamma = 1$) |
| 3.2 | Lichtafbuiging | $\alpha = 4GM/(bc^2)$ | Eddington 1919 |
| 3.3 | Baanprecissie | $\Delta\varphi = 3\pi r_s/(a(1-e^2))$ | Mercurius 42.98"/eeuw |
| 4.1 | Schwarzschildmetriek | $g_{tt}$, $g_{rr}$, $g_{\phi\phi}$ — alle drie exact | Formule (36)/(37) |
| 4.4 | Zijn in zwaartekracht | $S = m_0 \cdot c_{local}$ | Behoudswet |

**Vervolg**: Notebook 03 behandelt gevorderde onderwerpen (RN, Shapiro, geodetische precessie, fotonsfeer, GW, Kerr).